# Birth, Wealth, and Income

U.S. crude birth rate against real GDP per capita and household net
worth, from FRED.

In [1]:
import os, sys, pathlib
import pandas as pd
import matplotlib.pyplot as plt

# Shared site chart style (site_style.py in the repo root)
sys.path.insert(0, str(next(p for p in pathlib.Path.cwd().resolve().parents
                            if (p / "_quarto.yml").exists())))
import site_style as ss
ss.apply()

In [2]:
# Annual series from FRED. With FRED_API_KEY set, the data is pulled fresh and cached;
# otherwise the cached copy in data/ is used, so the page renders without a key.
SERIES = {
    'Birth_Rate_Per_1000': 'SPDYNCBRTINUSA',   # World Bank, crude birth rate
    'Real_GDP_Per_Capita': 'A939RX0Q048SBEA',  # BEA, chained 2017 dollars
    'Household_Net_Worth': 'HNONWRA027N',      # Federal Reserve Z.1, millions of nominal dollars
}
CACHE = pathlib.Path('data/fred_annual.csv')

if os.environ.get('FRED_API_KEY'):
    from fredapi import Fred
    fred = Fred(api_key=os.environ['FRED_API_KEY'])
    raw = pd.concat([fred.get_series(sid, observation_start='1945-01-01').rename(name)
                     for name, sid in SERIES.items()], axis=1, sort=True)
    annual = raw.resample('YE').mean()
    annual.index = annual.index.year.rename('year')
    CACHE.parent.mkdir(exist_ok=True)
    annual.to_csv(CACHE)

# Keep only years where all three series exist (the birth rate starts in 1960)
df_clean = pd.read_csv(CACHE, index_col='year').dropna()

In [3]:
from matplotlib.ticker import FuncFormatter

years = df_clean.index
births = df_clean['Birth_Rate_Per_1000']
panels = [
    # (heading, series, axis label, tick format)
    ('Birth rate vs. real income',
     df_clean['Real_GDP_Per_Capita'],
     'Real GDP per capita',
     FuncFormatter(lambda v, _: f'${v / 1e3:.0f}k' if v else '$0')),
    ('Birth rate vs. household net worth',
     df_clean['Household_Net_Worth'] / 1e6,   # FRED reports millions of USD
     'Household net worth',
     FuncFormatter(lambda v, _: f'${v:.0f}tn' if v else '$0')),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

for ax, (heading, series, label, fmt) in zip(axes, panels):
    ax.plot(years, births, color=ss.BRICK)
    ss.color_axis(ax, ss.BRICK)
    ss.axis_heading(ax, 'Births per 1,000 people', ss.BRICK, side='left')

    ax2 = ss.twin(ax)
    ax2.plot(years, series, color=ss.INDIGO)
    ax2.yaxis.set_major_formatter(fmt)
    ax2.set_ylim(bottom=0)
    ss.color_axis(ax2, ss.INDIGO)
    ss.axis_heading(ax2, label, ss.INDIGO, side='right')

    ax.set_title(heading, pad=34)
    ax.margins(x=0.01)

fig.subplots_adjust(wspace=0.32)
ss.titles(fig, 'As the U.S. grew richer, its birth rate fell',
          f'Crude birth rate against two measures of national prosperity, '
          f'{years.min()} to {years.max()}', y=1.08)
ss.source_note(fig, 'Source: FRED (World Bank; U.S. Bureau of Economic Analysis; '
                    'Federal Reserve Board). Annual averages. Net worth is in nominal dollars.')
plt.show()